# Classic Favouritism: PM₂.₅ Analysis

**Research question:** Do political leaders direct lower-pollution development to their birth regions?
We test this using PM₂.₅ (fine particulate matter) as the sole pollution outcome.

**Data:** ACAG SatPM V5.GL.06 annual PM₂.₅ at 0.01°×0.01° (van Donkelaar et al., 2021)  
**Treatment:** PLAD birth-region indicator (Bomprezzi et al., 2025)  
**Boundaries:** GADM v4.1 ADM2  

**Specification:**
$$\ln(\text{PM}_{2.5,it} + 0.01) = \beta \cdot \text{BirthRegion}_{it} + \alpha_i + \gamma_{ct} + \varepsilon_{it}$$

FE: ADM2 region + country×year | SE clustered by leader-period

**Sections:**
1. Setup  
2. Load PM₂.₅ panel  
3. Build analysis panel and treatment variables  
4. Main regressions (Lag 0 / 1 / 2)  
5. Democracy interaction  
6. Autocracy-only subsamples  
7. Event study

## 1. Setup

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

warnings.filterwarnings("ignore")

ROOT      = Path("..")           # project root
DATA      = ROOT / "data"
PLAD_PATH = DATA / "political leaders" / "PLAD_April_2024_gadm41.parquet"
SCRIPTS   = (ROOT / "scripts").resolve()
if str(SCRIPTS) not in sys.path:
    sys.path.append(str(SCRIPTS))

from gid_utils import valid_gid_mask

ANALYSIS_YEARS = range(1998, 2025)  # 1998-2024

print(f"Project root  : {ROOT.resolve()}")
print(f"Analysis years: {min(ANALYSIS_YEARS)}-{max(ANALYSIS_YEARS)}")

Project root  : /Users/richard/University/Political Leader Pollution
Analysis years: 1998-2024


## 2. Load PM₂.₅ panel (ACAG V5.GL.06)

In [2]:
PM25_PANEL_CACHE = DATA / "pm25_adm2_acag_panel.parquet"

assert PM25_PANEL_CACHE.exists(), (
    f"PM2.5 panel not found at {PM25_PANEL_CACHE}.\n"
    "Run scripts/build_pm25_panel.py first."
)

pm25_panel = pd.read_parquet(PM25_PANEL_CACHE)
pm25_panel = pm25_panel[pm25_panel["year"].isin(ANALYSIS_YEARS)].copy()
pm25_panel = pm25_panel.loc[valid_gid_mask(pm25_panel["GID_2"])].copy()
pm25_panel = pm25_panel.dropna(subset=["pm25_mean"])
pm25_panel = pm25_panel[pm25_panel["pm25_mean"] >= 0]
pm25_panel = pm25_panel[pm25_panel["pm25_mean"] < 1000]  # drop residual sentinel values

print(f"PM2.5 panel : {len(pm25_panel):,} obs | "
      f"{pm25_panel['GID_2'].nunique():,} regions | "
      f"years: {sorted(pm25_panel['year'].unique())}")
print(f"pm25_mean (ug/m3): mean={pm25_panel['pm25_mean'].mean():.2f}, "
      f"median={pm25_panel['pm25_mean'].median():.2f}, "
      f"max={pm25_panel['pm25_mean'].max():.1f}")
pm25_panel.head()

PM2.5 panel : 1,289,925 obs | 47,775 regions | years: [np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
pm25_mean (ug/m3): mean=17.59, median=14.87, max=127.5


,GID_2,GID_1,GID_0,year,pm25_mean
0,AFG.1.10_1,AFG.1_1,AFG,1998,11.694771
1,AFG.1.10_1,AFG.1_1,AFG,1999,14.878976
2,AFG.1.10_1,AFG.1_1,AFG,2000,18.561874
3,AFG.1.10_1,AFG.1_1,AFG,2001,22.568627
4,AFG.1.10_1,AFG.1_1,AFG,2002,20.218954


## 3. Build PLAD treatment variables and analysis panel

In [3]:
# Load and filter PLAD
plad = pd.read_parquet(PLAD_PATH)
plad = plad[plad["foreign_leader"] == "0"].copy()

BIRTH_GID = "gid_2" if "gid_2" in plad.columns else "gid_1"
plad = plad.loc[valid_gid_mask(plad[BIRTH_GID])].copy()
plad["startyear"] = plad["startyear"].astype(int)
plad["endyear"]   = plad["endyear"].astype(int)
plad = plad[plad["archigos_id"].str.strip() != "."].copy()
plad = plad.sort_values(["gid_0", "startyear"]).reset_index(drop=True)

print(f"Using birth region column : {BIRTH_GID}")
print(f"Leaders: {len(plad)} | countries: {plad['gid_0'].nunique()}")

Using birth region column : gid_2
Leaders: 820 | countries: 148


In [4]:
# Fix transition-year overlap
plad_fixed = plad.copy().reset_index(drop=True)
for gid_0, group in plad_fixed.groupby("gid_0"):
    idxs = group.index.tolist()
    for i in range(len(idxs) - 1):
        curr, nxt = idxs[i], idxs[i + 1]
        if plad_fixed.loc[curr, "endyear"] >= plad_fixed.loc[nxt, "startyear"]:
            plad_fixed.loc[curr, "endyear"] = plad_fixed.loc[nxt, "startyear"] - 1

# Expand spells into GID_2 x year rows
YEAR_MIN, YEAR_MAX = min(ANALYSIS_YEARS), max(ANALYSIS_YEARS)
rows = []
for idx, row in plad_fixed.iterrows():
    for y in range(max(int(row["startyear"]), YEAR_MIN),
                   min(int(row["endyear"]),   YEAR_MAX) + 1):
        rows.append({"GID_2": row[BIRTH_GID], "GID_0": row["gid_0"],
                     "year": y, "spell_id": idx})

leader_years = pd.DataFrame(rows)
leader_years = leader_years.loc[valid_gid_mask(leader_years["GID_2"])].copy()
leader_years = leader_years.drop_duplicates(subset=["GID_2", "year"])
leader_years["birth_region_leader"] = 1

# Country-year to spell mapping for SE clustering
spell_map = (leader_years[["GID_0", "year", "spell_id"]]
             .drop_duplicates(subset=["GID_0", "year"]))

print(f"Birth-district x year obs : {len(leader_years):,}")
print(f"Unique leader spells      : {spell_map['spell_id'].nunique():,}")
print(f"Year range in treatment   : {leader_years['year'].min()}-{leader_years['year'].max()}")

Birth-district x year obs : 3,103
Unique leader spells      : 532
Year range in treatment   : 1998-2023


In [5]:
# Merge PM2.5 + treatment
panel = pm25_panel.copy()

panel = panel.merge(
    leader_years[["GID_2", "year", "birth_region_leader"]],
    on=["GID_2", "year"], how="left"
)
panel["birth_region_leader"] = panel["birth_region_leader"].fillna(0).astype(int)

panel = panel.merge(
    spell_map[["GID_0", "year", "spell_id"]], on=["GID_0", "year"], how="left"
)
panel["spell_id"] = panel["spell_id"].fillna(
    panel["GID_0"] + "_" + panel["year"].astype(str) + "_noleader"
).astype(str)

# Lag variables
panel = panel.sort_values(["GID_2", "year"])
panel["brl_lag1"] = panel.groupby("GID_2")["birth_region_leader"].shift(1).fillna(0).astype(int)
panel["brl_lag2"] = panel.groupby("GID_2")["birth_region_leader"].shift(2).fillna(0).astype(int)

# Log PM2.5
panel["ln_pm25"] = np.log(panel["pm25_mean"] + 0.01)

# Country x year FE identifier
panel["country_year"] = panel["GID_0"] + "_" + panel["year"].astype(str)

panel = panel.dropna(subset=["pm25_mean"])
panel = panel.loc[valid_gid_mask(panel["GID_2"])].copy()

print(f"Analysis panel : {len(panel):,} obs | "
      f"{panel['GID_2'].nunique():,} regions | "
      f"years: {sorted(panel['year'].unique())}")
print(f"Treated obs (Lag 0) : {panel['birth_region_leader'].sum():,}")
print(f"Treated obs (Lag 1) : {panel['brl_lag1'].sum():,}")
print(f"Treated obs (Lag 2) : {panel['brl_lag2'].sum():,}")
panel.head()

Analysis panel : 1,289,925 obs | 47,775 regions | years: [np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Treated obs (Lag 0) : 2,853
Treated obs (Lag 1) : 2,853
Treated obs (Lag 2) : 2,816


,GID_2,GID_1,GID_0,year,pm25_mean,birth_region_leader,spell_id,brl_lag1,brl_lag2,ln_pm25,country_year
0,AFG.1.10_1,AFG.1_1,AFG,1998,11.694771,0,2.0,0,0,2.459997,AFG_1998
1,AFG.1.10_1,AFG.1_1,AFG,1999,14.878976,0,2.0,0,0,2.700621,AFG_1999
2,AFG.1.10_1,AFG.1_1,AFG,2000,18.561874,0,2.0,0,0,2.921648,AFG_2000
3,AFG.1.10_1,AFG.1_1,AFG,2001,22.568627,0,3.0,0,0,3.117004,AFG_2001
4,AFG.1.10_1,AFG.1_1,AFG,2002,20.218954,0,3.0,0,0,3.007115,AFG_2002


## 4. Main Regressions — Classic Favouritism (PM₂.₅)

$$\ln(\text{PM}_{2.5,it} + 0.01) = \beta \cdot \text{BirthRegion}_{it} + \alpha_i + \gamma_{ct} + \varepsilon_{it}$$

**Prediction:** $\hat{\beta} < 0$ — birth regions receive lower PM₂.₅ (cleaner development).  
FE: ADM2 region + country×year | SE clustered by leader-period

In [6]:
panel_idx = panel.set_index(["GID_2", "year"])

specs = [
    ("Lag 0", "birth_region_leader"),
    ("Lag 1", "brl_lag1"),
    ("Lag 2", "brl_lag2"),
]

rows = []
for lag_label, var in specs:
    model = PanelOLS.from_formula(
        f"ln_pm25 ~ {var} + EntityEffects",
        data=panel_idx,
        other_effects=panel_idx["country_year"],
        drop_absorbed=True,
    )
    res   = model.fit(cov_type="clustered", clusters=panel_idx["spell_id"])
    coef  = res.params[var]
    se    = res.std_errors[var]
    pval  = res.pvalues[var]
    stars = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
    rows.append({
        "Spec":      lag_label,
        "Coeff":     f"{coef:.4f}{stars}",
        "Std. Err.": f"({se:.4f})",
        "p-value":   f"{pval:.3f}",
        "N":         f"{res.nobs:,}",
    })

results_df = pd.DataFrame(rows).set_index("Spec")
print("Classic Favouritism - PM2.5")
print("Dep. var: log(PM2.5 + 0.01)")
print("FE: Region (ADM2) + Country x Year  |  Clustering: Leader-period\n")
print(results_df.to_string())
print("\nSignificance: *** p<0.01, ** p<0.05, * p<0.1")
print("\nFavourtism prediction: coefficient < 0 (lower PM2.5 in birth regions)")

Classic Favouritism - PM2.5
Dep. var: log(PM2.5 + 0.01)
FE: Region (ADM2) + Country x Year  |  Clustering: Leader-period

           Coeff Std. Err. p-value          N
Spec                                         
Lag 0   0.0063**  (0.0027)   0.018  1,289,925
Lag 1  0.0063***  (0.0023)   0.006  1,289,925
Lag 2  0.0062***  (0.0022)   0.005  1,289,925

Significance: *** p<0.01, ** p<0.05, * p<0.1

Favourtism prediction: coefficient < 0 (lower PM2.5 in birth regions)


## 5. Democracy Interaction

Hodler & Raschky's core result is strongest in autocracies. Interact `birth_region_leader`
with V-Dem `v2x_polyarchy` (0 = full autocracy, 1 = full democracy).

- `BirthRegion`: treatment effect in a full autocracy (`democracy = 0`)
- `BirthRegion x Democracy`: how the effect changes as democracy rises

Prediction: `BirthRegion` < 0 and `BirthRegion x Democracy` > 0

In [7]:
VDEM_DIR = DATA / "vdem"
vdem_csv = list(VDEM_DIR.glob("*.csv")) if VDEM_DIR.exists() else []
assert vdem_csv, (
    "V-Dem CSV not found. Download the Country-Year Core CSV from "
    "https://v-dem.net/data/the-v-dem-dataset/ and place it in data/vdem/"
)

vdem = pd.read_csv(vdem_csv[0], low_memory=False)
vdem = vdem[["country_text_id", "year", "v2x_polyarchy"]].rename(
    columns={"country_text_id": "GID_0", "v2x_polyarchy": "democracy"}
)
vdem = vdem.dropna(subset=["democracy"])
vdem = vdem[vdem["year"].isin(ANALYSIS_YEARS)].copy()

panel_dem = panel.merge(vdem, on=["GID_0", "year"], how="left")
panel_dem = panel_dem.dropna(subset=["democracy"])

panel_dem["birth_x_democracy"]    = panel_dem["birth_region_leader"] * panel_dem["democracy"]
panel_dem["brl_lag1_x_democracy"] = panel_dem["brl_lag1"]            * panel_dem["democracy"]
panel_dem["brl_lag2_x_democracy"] = panel_dem["brl_lag2"]            * panel_dem["democracy"]

print(f"V-Dem: {len(vdem):,} country-year obs | {vdem['GID_0'].nunique():,} countries")
print(f"Panel with democracy: {len(panel_dem):,} obs | {panel_dem['GID_2'].nunique():,} regions")

V-Dem: 4,810 country-year obs | 179 countries
Panel with democracy: 1,264,635 obs | 46,860 regions


In [8]:
panel_dem_idx = panel_dem.set_index(["GID_2", "year"])

specs_dem = [
    ("Lag 0", "birth_region_leader", "birth_x_democracy"),
    ("Lag 1", "brl_lag1",            "brl_lag1_x_democracy"),
    ("Lag 2", "brl_lag2",            "brl_lag2_x_democracy"),
]

rows_dem = []
for lag_label, var, inter in specs_dem:
    model = PanelOLS.from_formula(
        f"ln_pm25 ~ {var} + {inter} + EntityEffects",
        data=panel_dem_idx,
        other_effects=panel_dem_idx["country_year"],
        drop_absorbed=True,
    )
    res = model.fit(cov_type="clustered", clusters=panel_dem_idx["spell_id"])

    for label, key in [("BirthRegion", var), ("BirthRegion x Democracy", inter)]:
        coef  = res.params[key]
        se    = res.std_errors[key]
        pval  = res.pvalues[key]
        stars = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
        rows_dem.append({
            "Spec":     lag_label,
            "Variable": label,
            "Coeff":    f"{coef:.4f}{stars}",
            "SE":       f"({se:.4f})",
            "p":        f"{pval:.3f}",
            "N":        f"{res.nobs:,}",
        })

results_dem = pd.DataFrame(rows_dem).set_index(["Spec", "Variable"])
print("Democracy Interaction - PM2.5")
print("Dep. var: log(PM2.5 + 0.01)")
print("FE: Region (ADM2) + Country x Year  |  Clustering: Leader-period\n")
print(results_dem.to_string())
print("\nInterpretation: BirthRegion = effect in full autocracy (democracy = 0).")

Democracy Interaction - PM2.5
Dep. var: log(PM2.5 + 0.01)
FE: Region (ADM2) + Country x Year  |  Clustering: Leader-period

                                Coeff        SE      p          N
Spec  Variable                                                   
Lag 0 BirthRegion              0.0039  (0.0069)  0.569  1,264,635
      BirthRegion x Democracy  0.0042  (0.0108)  0.696  1,264,635
Lag 1 BirthRegion              0.0040  (0.0063)  0.520  1,264,635
      BirthRegion x Democracy  0.0040  (0.0096)  0.676  1,264,635
Lag 2 BirthRegion              0.0052  (0.0063)  0.410  1,264,635
      BirthRegion x Democracy  0.0018  (0.0095)  0.851  1,264,635

Interpretation: BirthRegion = effect in full autocracy (democracy = 0).


## 6. Autocracy-Only Regressions

Restrict sample by V-Dem `v2x_polyarchy` thresholds. If favouritism operates primarily
through autocratic channels, the negative PM₂.₅ effect should be strongest here.

In [9]:
autocracy_thresholds = [0.3, 0.5]
specs_auto = [
    ("Lag 0", "birth_region_leader"),
    ("Lag 1", "brl_lag1"),
    ("Lag 2", "brl_lag2"),
]

rows_auto = []
for thr in autocracy_thresholds:
    sub     = panel_dem[panel_dem["democracy"] < thr].copy()
    sub_idx = sub.set_index(["GID_2", "year"])
    print(f"\nAutocracy sample (democracy < {thr}): "
          f"{len(sub):,} obs | {sub['GID_2'].nunique():,} regions | "
          f"treated: {sub['birth_region_leader'].sum():,}")

    for lag_label, var in specs_auto:
        model = PanelOLS.from_formula(
            f"ln_pm25 ~ {var} + EntityEffects",
            data=sub_idx,
            other_effects=sub_idx["country_year"],
            drop_absorbed=True,
        )
        res   = model.fit(cov_type="clustered", clusters=sub_idx["spell_id"])
        coef  = res.params[var]
        se    = res.std_errors[var]
        pval  = res.pvalues[var]
        stars = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
        rows_auto.append({
            "Threshold": f"democracy < {thr}",
            "Spec":      lag_label,
            "Coeff":     f"{coef:.4f}{stars}",
            "SE":        f"({se:.4f})",
            "p":         f"{pval:.3f}",
            "N":         f"{res.nobs:,}",
        })

auto_df = pd.DataFrame(rows_auto).set_index(["Threshold", "Spec"])
print("\nAutocracy-only estimates")
print("Dep. var: log(PM2.5 + 0.01)")
print("FE: Region (ADM2) + Country x Year  |  Clustering: Leader-period\n")
print(auto_df.to_string())


Autocracy sample (democracy < 0.3): 212,754 obs | 15,113 regions | treated: 869

Autocracy sample (democracy < 0.5): 420,189 obs | 22,267 regions | treated: 1,406

Autocracy-only estimates
Dep. var: log(PM2.5 + 0.01)
FE: Region (ADM2) + Country x Year  |  Clustering: Leader-period

                        Coeff        SE      p        N
Threshold       Spec                                   
democracy < 0.3 Lag 0  0.0030  (0.0076)  0.688  212,754
                Lag 1  0.0051  (0.0071)  0.468  212,754
                Lag 2  0.0068  (0.0076)  0.372  212,754
democracy < 0.5 Lag 0  0.0026  (0.0047)  0.589  420,189
                Lag 1  0.0024  (0.0041)  0.563  420,189
                Lag 2  0.0032  (0.0040)  0.421  420,189


## 7. Event Study

Dynamic specification around each district's **first treated year** in the sample.  
Event window: $k \in [-5, 5]$, $k = -1$ omitted as the reference period.

- **Pre-trends** ($k < 0$): should be close to zero for a valid design
- **Post-treatment** ($k \geq 0$): negative = lower PM₂.₅ after leader ascension

In [10]:
panel_es = panel.copy()

# First treatment year per district
first_treat = (
    panel_es.loc[panel_es["birth_region_leader"] == 1]
    .groupby("GID_2")["year"]
    .min()
    .rename("first_treat_year")
)
panel_es = panel_es.merge(first_treat, on="GID_2", how="left")
panel_es["event_time"] = panel_es["year"] - panel_es["first_treat_year"]

# Binary indicators for each event-time bin (omit k = -1)
event_ks   = [k for k in range(-5, 6) if k != -1]
event_vars = []
for k in event_ks:
    name = f"ev_{'m' if k < 0 else 'p'}{abs(k)}"
    panel_es[name] = (panel_es["event_time"] == k).astype(int)
    event_vars.append((k, name))

panel_es_idx = panel_es.set_index(["GID_2", "year"])
rhs = " + ".join([v for _, v in event_vars])

model_es = PanelOLS.from_formula(
    f"ln_pm25 ~ {rhs} + EntityEffects",
    data=panel_es_idx,
    other_effects=panel_es_idx["country_year"],
    drop_absorbed=True,
)
res_es = model_es.fit(cov_type="clustered", clusters=panel_es_idx["spell_id"])

rows_es = []
for k, var in event_vars:
    if var not in res_es.params.index:
        continue
    coef  = res_es.params[var]
    se    = res_es.std_errors[var]
    pval  = res_es.pvalues[var]
    stars = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
    rows_es.append({
        "k":     k,
        "Coeff": f"{coef:.4f}{stars}",
        "SE":    f"({se:.4f})",
        "CI_lo": f"{coef - 1.96*se:.4f}",
        "CI_hi": f"{coef + 1.96*se:.4f}",
        "p":     f"{pval:.3f}",
    })

es_df = pd.DataFrame(rows_es).set_index("k")
print("Event Study - PM2.5")
print("Dep. var: log(PM2.5 + 0.01) | Reference period: k = -1")
print("FE: Region (ADM2) + Country x Year  |  Clustering: Leader-period\n")
print(es_df.to_string())
print("\nPre-trends (k<0) close to zero = parallel trends supported")
print("Post-treatment (k>=0) negative  = lower PM2.5 in birth regions")

Event Study - PM2.5
Dep. var: log(PM2.5 + 0.01) | Reference period: k = -1
FE: Region (ADM2) + Country x Year  |  Clustering: Leader-period

      Coeff        SE    CI_lo   CI_hi      p
k                                            
-5  -0.0036  (0.0052)  -0.0139  0.0066  0.488
-4  -0.0072  (0.0055)  -0.0179  0.0035  0.186
-3  -0.0040  (0.0051)  -0.0140  0.0059  0.430
-2  -0.0012  (0.0041)  -0.0092  0.0068  0.771
 0   0.0043  (0.0045)  -0.0045  0.0131  0.340
 1   0.0045  (0.0040)  -0.0034  0.0124  0.269
 2  -0.0010  (0.0035)  -0.0077  0.0058  0.780
 3  -0.0024  (0.0032)  -0.0087  0.0040  0.464
 4   0.0041  (0.0031)  -0.0019  0.0102  0.182
 5   0.0007  (0.0032)  -0.0055  0.0069  0.821

Pre-trends (k<0) close to zero = parallel trends supported
Post-treatment (k>=0) negative  = lower PM2.5 in birth regions
